# Isochronous topopt II -- Sculpt + HDiv-MMM shape regeneration

[`isochronous_topopt.ipynb`](isochronous_topopt.ipynb) ends with a *density* design on the
sector pole: a per-element gray field on the design mesh.  This notebook takes that design
the rest of the way to a **manufacturable body** and closes the loop numerically:

1. **Design** -- the same 30-iterate SLP loop on the 194-tet sector (`DensityAdjointVIM`,
   build-once charge Gram, $\chi=1000$), verified by the exact-void iron-only extraction.
2. **Shape regeneration** (`radia.topopt_cad`) -- volume-weighted $P_0\to P_1$ nodal level
   set, marching-cubes iso-surface, watertight STL (capped Taubin smoothing + quadric
   decimation, every drift measured).
3. **Sculpt all-hex meshing** (`cubit_stl_to_vol`, Coreform Cubit headless batch) --
   the overlay-grid mesher lays a background grid over the STL and cuts it, so it needs
   **no sweepable topology** and **does not inherit the STL facet layout** -- exactly right
   for topology-optimized blobs.
4. **Re-evaluation with the same operator** -- the regenerated body's objective is computed
   by the same HDiv-MMM charge-Gram machinery, iron-only, **no air mesh at any stage**
   (Radia's analytic open boundary), and compared against the staircase iron extraction.

The pairing is the point: Sculpt hex-meshes arbitrary-topology bodies in seconds, and
HDiv-MMM only ever needs the iron volume -- so a shape that changes every design cycle
never triggers air re-meshing or boundary truncation.

**Requirements**: `radia`, `ngsolve`, `trimesh`, `scikit-image`, and a Coreform Cubit
license (Sculpt ships in the Windows `bin/` too; everything runs `-batch -nographics`).
The heavy cell is the regenerated-body Gram build (~4 min at 1.5 k hexes).

In [1]:
import json
import time
from math import cos, pi, sin
from pathlib import Path

import numpy as np
from netgen.occ import Cylinder, HalfSpace, OCCGeometry, Pnt, Vec, Z
from ngsolve import CoefficientFunction, HDiv, InnerProduct, Mesh, SetNumThreads, TaskManager

from radia.isochronous_topopt import (
    MU0, DensityAdjointVIM, HelmholtzFilter, density_to_s,
    field_functional_load, gradient_pair_points, optimize_density,
    orbit_arc_points, uniform_field_load, verify_design_iron_only)
from radia.topopt_cad import iso_stl_from_grid, nodal_from_element_density

SetNumThreads(4)
OUT = Path("topopt_shape_regen_work").resolve()
OUT.mkdir(exist_ok=True)


def make_sector_mesh(R1=0.05, R2=0.15, angle_deg=60.0, z0=0.02, thick=0.03, maxh=0.02):
    ang = angle_deg * pi / 180.0
    ring = (Cylinder(Pnt(0, 0, z0), Z, r=R2, h=thick)
            - Cylinder(Pnt(0, 0, z0 - 0.01), Z, r=R1, h=thick + 0.02))
    hs1 = HalfSpace(Pnt(0, 0, 0), Vec(0, -1, 0))
    hs2 = HalfSpace(Pnt(0, 0, 0), Vec(-sin(ang), cos(ang), 0))
    return Mesh(OCCGeometry(ring * hs1 * hs2).GenerateMesh(maxh=maxh))


CHI_IRON = 1000.0
SPAN = (pi / 12, pi / 2 - pi / 12)
obj_pts, obj_radial = orbit_arc_points(0.115, 0.0, 7, span=SPAN)
pair_pts, pair_wts = gradient_pair_points(
    obj_pts, np.full(7, 1 / 7), delta=0.01, direction=obj_radial)


def state_builder(fes):
    return uniform_field_load(fes, (0.0, 0.0, 1.0e5))


def objective_builder(fes):
    return field_functional_load(fes, pair_pts, pair_wts, axis=2,
                                 scale=MU0, bonus_intorder=10)


def constraint_builder(radius):
    def build(fes):
        cpts, _ = orbit_arc_points(radius, 0.0, 7, span=SPAN)
        return field_functional_load(fes, cpts, np.full(7, 1 / 7), axis=2,
                                     scale=MU0, bonus_intorder=10)
    return build


con_builders = [constraint_builder(r) for r in (0.08, 0.10)]
with TaskManager():
    mesh = make_sector_mesh()
    fes = HDiv(mesh, order=1)
    prob = DensityAdjointVIM(fes, eps=1e-7)
    f_state = state_builder(fes)
    cons = [b(fes) for b in con_builders]
    filt = HelmholtzFilter(mesh, radius=0.012)
    lin0 = prob.linearize(
        density_to_s(filt.apply(np.full(prob.n_el, 0.5)), CHI_IRON),
        f_state, [objective_builder(fes)] + cons)
    targets = [float(v) for v in lin0.values[1:]]
    t0 = time.perf_counter()
    result = optimize_density(prob, f_state, objective_builder(fes), cons,
                              targets, chi_iron=CHI_IRON, volume_fraction=0.5,
                              density_filter=filt, move_limit=0.1,
                              max_iterations=30)
    verification = verify_design_iron_only(
        prob, result.density, state_builder,
        [objective_builder] + con_builders, chi_iron=CHI_IRON,
        density_filter=filt, gram_kwargs=dict(eps=1e-7))
J0 = float(result.history[0]["objective"])
J1 = float(result.history[-1]["objective"])
print(f"design: ne={mesh.ne}  iterates={len(result.history)}  "
      f"wall={time.perf_counter()-t0:.1f}s")
print(f"J {J0:+.6e} -> {J1:+.6e}  ({100*(J1-J0)/abs(J0):+.2f}%)")
print(f"iron-only extraction: {verification.iron_mesh.ne} elements, "
      f"ersatz bands {[float(f'{b:+.4f}') for b in np.atleast_1d(verification.bands)]}")

design: ne=194  iterates=5  wall=3.1s
J +2.758902e-02 -> +2.943788e-02  (+6.70%)
iron-only extraction: 72 elements, ersatz bands [0.0461, -0.0975, -0.277]


## Density to a watertight STL

`nodal_from_element_density` lifts the filtered $P_0$ density to vertices by
volume-weighted averaging; `iso_stl_from_grid` resamples it on a Cartesian grid
(linear Delaunay interpolation, void outside a distance cutoff, one-cell Gaussian blur),
runs marching cubes at $\rho=0.5$, and returns a **watertight** surface or raises.
Two measured levers matter:

* **Taubin smoothing** is capped at 5 iterations (10 iterations once shrank a body 12.8 %);
  every drift is returned, not assumed.
* **Quadric decimation** (`target_faces`) controls the facet count.  A tet mesher conforms
  to the facets, so facet density becomes solver-mesh density; Sculpt does not care.

`cubit_stl_to_vol` then meshes the STL headlessly and gates the result: volume closure vs
the STL, zero inverted elements (gmsh `getJacobians`), and **boundary faces present**
(`.vol` skin == `.msh` topological skin).  The boundary gate exists because a bare Sculpt
free mesh once exported *zero* surface elements -- the charge Gram silently lost every
surface charge and returned the demag-free field, 51x off (bug-pattern
`sculpt-free-mesh-vol-zero-boundary-faces`; now also fixed in the exporter and raised on
in `vim.build_charge_gram`).

In [2]:
from radia_mcp.cubit.server import cubit_stl_to_vol

rho_f = np.clip(filt.apply(result.density), 0.0, 1.0)
nodal = nodal_from_element_density(mesh, rho_f)
stl = OUT / "design_body.stl"
stl_info = iso_stl_from_grid(mesh, nodal, stl, level=0.5, resolution=96,
                             smooth_iterations=3, target_faces=1500)
print(f"iso STL: {stl_info['n_faces']} faces, watertight={stl_info['watertight']}, "
      f"V={stl_info['volume']:.4e} m^3")
print(f"  smoothing drift {100*stl_info['smoothing_volume_drift']:+.2f}%   "
      f"decimation drift {100*stl_info['decimation_volume_drift']:+.2f}%")

r = json.loads(cubit_stl_to_vol(
    stl_path=str(stl), scheme="hex", closure_tolerance=0.08,
    out_vol=str(OUT / "regen_hex.vol"), out_msh=str(OUT / "regen_hex.msh")))
assert r["status"] == "ok", r
print(f"Sculpt hex: {r['by_type'][0]['n']} elements, gates={r['gates']}")
print(f"  closure {100*r['closure']:+.2f}%   boundary faces "
      f"{r['vol_boundary_faces']} == skin {r['msh_skin_faces']}")

iso STL: 1500 faces, watertight=True, V=1.2782e-04 m^3
  smoothing drift -0.18%   decimation drift -0.18%


Sculpt hex: 1537 elements, gates={'closure_ok': True, 'no_inverted_elements': True, 'boundary_faces_ok': True}
  closure +2.15%   boundary faces 738 == skin 738


## Re-evaluating the manufactured shape with the same operator

Both bodies -- the staircase iron-only extraction and the regenerated smooth hex body --
are evaluated by the same machinery at $\chi_\mathrm{eval}=100$: one `DensityAdjointVIM`
per mesh, uniform $s=1/\chi$, `solver="native"` (batched mass-Riesz PCG in C++).  The
`DemagFactor` printed below is the Rayleigh quotient of the charge Gram for uniform
$\hat z$ magnetization: a flat sector reads about $0.5$; a broken boundary export reads
exactly $0$.

**Why hex and not tet for this evaluation?**  The facet-conforming tet mesh of the same
decimated STL carries micro-cell blobs that make the assembled charge Gram *indefinite*
(measured smallest generalized eigenvalue $\mu_{\min}=-2.67$ against the physical
spectrum $[0,1]$), and CG then stalls at any tolerance and preconditioner -- see
bug-pattern `facet-tet-charge-gram-indefinite-cg-stall` and the validation lane
`validation_test/isochronous_topopt/test_shape_regen_lane.py`.  The Sculpt hex mesh of
the same surface is PSD-behaved and solves in ~20 iterations.

In [3]:
CHI_EVAL = 100.0
with TaskManager():
    m_stair = verification.iron_mesh
    fes_stair = HDiv(m_stair, order=1)
    prob_stair = DensityAdjointVIM(fes_stair, eps=1e-7)
    lin_stair = prob_stair.linearize(
        np.full(prob_stair.n_el, 1.0 / CHI_EVAL), state_builder(fes_stair),
        [objective_builder(fes_stair)], tol=1e-10, solver="native")
    t0 = time.perf_counter()
    m_regen = Mesh(str(OUT / "regen_hex.vol"))
    fes_regen = HDiv(m_regen, order=1)
    prob_regen = DensityAdjointVIM(fes_regen, eps=1e-7)
    t_gram = time.perf_counter() - t0
    demag_factor_z = float(prob_regen.demag.DemagFactor(
        CoefficientFunction((0.0, 0.0, 1.0))))
    lin_regen = prob_regen.linearize(
        np.full(prob_regen.n_el, 1.0 / CHI_EVAL), state_builder(fes_regen),
        [objective_builder(fes_regen)], tol=1e-10, solver="native")
J_stair = float(lin_stair.values[0])
J_regen = float(lin_regen.values[0])
delta = (J_regen - J_stair) / abs(J_stair)
print(f"regen hex Gram build: {t_gram:.0f}s at {m_regen.ne} elements "
      f"({fes_regen.ndof} dofs)")
print(f"DemagFactor(z) = {demag_factor_z:+.4f}   (flat sector ~0.5; broken export = 0)")
print(f"J staircase ({m_stair.ne:5d} tets) = {J_stair:+.6e}   "
      f"[{lin_stair.state_iterations}/{lin_stair.adjoint_iterations} its]")
print(f"J regenerated ({m_regen.ne:4d} hexes) = {J_regen:+.6e}   "
      f"[{lin_regen.state_iterations}/{lin_regen.adjoint_iterations} its]")
print(f"smooth-vs-staircase delta = {100*delta:+.2f}%")
assert demag_factor_z > 0.2 and abs(delta) < 0.5

regen hex Gram build: 223s at 1537 elements (38364 dofs)
DemagFactor(z) = +0.4775   (flat sector ~0.5; broken export = 0)
J staircase (   72 tets) = +7.196639e-01   [37/(50,) its]
J regenerated (1537 hexes) = +7.354968e-01   [21/(25,) its]
smooth-vs-staircase delta = +2.20%


## The regenerated body and its magnetization

The interactive scenes below show the Sculpt hex mesh of the manufactured shape and the
HDiv magnetization $\mathbf{M}$ from the $\chi_\mathrm{eval}=100$ state solve (the field
whose functional was compared above).  The body is meshed alone -- there is no air region
to show, which is the whole point of the charge-Gram route.

In [4]:
from ngsolve.webgui import Draw

Draw(m_regen)
Draw(lin_regen.gfM, m_regen, name="M_regen",
     vectors={"grid_size": 30}, autoscale=True,
     clipping={"x": 0.0, "y": 0.0, "z": -1.0, "dist": 0.0206})

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

## Where this goes

* The executable regression for this whole pipeline (including the boundary-face and
  indefiniteness locks, three mesh families, solver parity, and reciprocity checks) is
  `validation_test/isochronous_topopt/test_shape_regen_lane.py`.
* Because the SIMP density grid and Sculpt's overlay grid belong to the same geometric
  family, the design loop itself can run on Sculpt hexes (the RT1-hex charge Gram ships
  in the radia wheel) -- making design representation and manufacturing mesh one object.
  A first full-hex prototype run closes end-to-end: a 360-hex Sculpt design domain
  (min quality 0.54) converged in five SLP iterates (+3.3 % objective at that coarse
  resolution), the hex iron-only extraction verified it, and the regenerated 1.4 k-hex
  body re-evaluated within +5.7 % of the staircase value.
* The hex charge-Gram build cost is the current optimization target: measured 237 s at
  1537 elements, 99.8 % of it in the C++ Gram fill (the Python charge basis takes
  0.4 s); relaxing the ACA tolerance to 1e-4 buys only 27 %, so the real lever is a
  near/far quadrature split in the hex fill like the tet path already has.  The tet
  route remains available as a *mesh product* but is not used for charge-Gram
  re-evaluation (see the bug patterns referenced above).